# KSE-30 Feature Engineering
### Technical Indicators + Feature Selection
---
**Pipeline:**
1. Load Clean Data
2. Technical Indicators (RSI, MACD, Bollinger Bands, Momentum, Volatility, RoC)
3. Visualise All Indicators
4. Feature Selection — CFS (Correlation-Based)
5. Feature Selection — RFE with Random Forest
6. Final Feature Set
7. Save + Download

## 0. Install & Import

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn scipy ta -q
print('Done.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
import pickle
import glob
import warnings
warnings.filterwarnings('ignore')

# ── Global plot settings ─────────────────────────────────────────────────────
plt.rcParams['figure.dpi']      = 300
plt.rcParams['savefig.dpi']     = 300
plt.rcParams['figure.figsize']  = (16, 5)
plt.rcParams['axes.grid']       = True
plt.rcParams['grid.alpha']      = 0.3
plt.rcParams['font.size']       = 9
sns.set_style('whitegrid')

SECTOR_MAP = {
    'ABOT.KA':'Pharmaceuticals', 'BAFL.KA':'Banking', 'DGKC.KA':'Cement',
    'EFERT.KA':'Fertilizer', 'FCCL.KA':'Cement', 'FFC.KA':'Fertilizer',
    'HBL.KA':'Banking', 'HUBC.KA':'Power_Generation', 'ICI.KA':'Chemicals',
    'KAPCO.KA':'Power_Generation', 'KEL.KA':'Power_Generation', 'KOHC.KA':'Cement',
    'LUCK.KA':'Cement', 'MARI.KA':'Oil_Gas', 'MCB.KA':'Banking', 'MEBL.KA':'Banking',
    'MLCF.KA':'Cement', 'NBP.KA':'Banking', 'NESTLE.KA':'Food_Beverages',
    'OGDC.KA':'Oil_Gas', 'PKGS.KA':'Paper_Board', 'POL.KA':'Oil_Gas',
    'PPL.KA':'Oil_Gas', 'PSEL.KA':'Financial_Services', 'PSO.KA':'Oil_Gas',
    'SEARL.KA':'Pharmaceuticals', 'SYS.KA':'Technology', 'TRG.KA':'Technology',
    'UBL.KA':'Banking'
}

print('Setup complete.')

---
## 1. Load Clean Data

In [ ]:
from google.colab import files
print('Upload these files:')
print('  1. KSE30_stocks_clean.csv')
print('  2. KSE30_benchmarks_clean_wide.csv')
uploaded = files.upload()

fnames = list(uploaded.keys())
def find(kw):
    for f in fnames:
        if kw.lower() in f.lower(): return f
    return None

df = pd.read_csv(find('stocks_clean'))
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Ticker','Date']).reset_index(drop=True)
df['Sector'] = df['Ticker'].map(SECTOR_MAP)

bench_wide = pd.read_csv(find('wide'))
bench_wide['Date'] = pd.to_datetime(bench_wide['Date'])
bench_wide = bench_wide.sort_values('Date').set_index('Date')

print(f'Stocks : {df["Ticker"].nunique()} tickers, {len(df):,} rows')
print(f'Bench  : {bench_wide.shape}')
print(f'Columns: {list(df.columns)}')

---
## 2. Technical Indicator Functions
All indicators are computed **per stock** using only price/volume data.

In [ ]:
# ── Indicator helper functions ────────────────────────────────────────────────

def compute_rsi(close, period=14):
    """Relative Strength Index — measures momentum (overbought/oversold)"""
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(close, fast=12, slow=26, signal=9):
    """MACD — trend-following momentum indicator"""
    ema_fast   = close.ewm(span=fast, adjust=False).mean()
    ema_slow   = close.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line= macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return macd_line, signal_line, histogram


def compute_bollinger(close, period=20, std_dev=2):
    """Bollinger Bands — volatility and mean-reversion indicator"""
    sma        = close.rolling(period).mean()
    std        = close.rolling(period).std()
    upper_band = sma + std_dev * std
    lower_band = sma - std_dev * std
    bb_width   = (upper_band - lower_band) / sma      # bandwidth
    bb_pct     = (close - lower_band) / (upper_band - lower_band)  # %B
    return upper_band, sma, lower_band, bb_width, bb_pct


def compute_momentum(close):
    """Price momentum over 3, 6, 12 month horizons (63, 126, 252 trading days)"""
    mom_3m  = close.pct_change(63)
    mom_6m  = close.pct_change(126)
    mom_12m = close.pct_change(252)
    return mom_3m, mom_6m, mom_12m


def compute_volatility(close, short=10, medium=21, long=63):
    """Rolling historical volatility — annualized std of log returns"""
    log_ret  = np.log(close / close.shift(1))
    vol_10d  = log_ret.rolling(short).std()  * np.sqrt(252)
    vol_21d  = log_ret.rolling(medium).std() * np.sqrt(252)
    vol_63d  = log_ret.rolling(long).std()   * np.sqrt(252)
    return vol_10d, vol_21d, vol_63d


def compute_roc(close, periods=[5, 10, 21]):
    """Rate of Change — percentage price change over N periods"""
    rocs = {}
    for p in periods:
        rocs[f'RoC_{p}d'] = close.pct_change(p) * 100
    return rocs


def compute_atr(high, low, close, period=14):
    """Average True Range — measures price volatility range"""
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()


def compute_obv(close, volume):
    """On-Balance Volume — volume momentum indicator"""
    direction = np.sign(close.diff())
    return (direction * volume).cumsum()


print('All indicator functions defined.')

---
## 3. Compute All Indicators — Per Stock

In [ ]:
# ── Compute all indicators per stock ─────────────────────────────────────────
results = []

for ticker, grp in df.groupby('Ticker'):
    g = grp.copy().sort_values('Date').reset_index(drop=True)
    close  = g['Close']
    high   = g['High']
    low    = g['Low']
    volume = g['Volume']

    # RSI
    g['RSI_14'] = compute_rsi(close, 14)

    # MACD
    g['MACD'], g['MACD_Signal'], g['MACD_Hist'] = compute_macd(close)

    # Bollinger Bands
    g['BB_Upper'], g['BB_Mid'], g['BB_Lower'], g['BB_Width'], g['BB_Pct'] = \
        compute_bollinger(close)

    # Momentum
    g['Mom_3M'], g['Mom_6M'], g['Mom_12M'] = compute_momentum(close)

    # Volatility
    g['Vol_10d'], g['Vol_21d'], g['Vol_63d'] = compute_volatility(close)

    # Rate of Change
    rocs = compute_roc(close, [5, 10, 21])
    for k, v in rocs.items():
        g[k] = v

    # ATR
    g['ATR_14'] = compute_atr(high, low, close)

    # OBV
    g['OBV'] = compute_obv(close, volume)

    # Moving Averages (for reference / additional features)
    g['SMA_20']  = close.rolling(20).mean()
    g['SMA_50']  = close.rolling(50).mean()
    g['EMA_12']  = close.ewm(span=12, adjust=False).mean()
    g['EMA_26']  = close.ewm(span=26, adjust=False).mean()

    # Price relative to moving averages
    g['Price_SMA20_Ratio'] = close / g['SMA_20']
    g['Price_SMA50_Ratio'] = close / g['SMA_50']

    # Log return (target variable for model)
    g['Log_Return']         = np.log(close / close.shift(1))
    g['Forward_Log_Return'] = g['Log_Return'].shift(-1)  # next-day return = target

    results.append(g)

df_feat = pd.concat(results, ignore_index=True)

# Drop rows where target is NaN (last row per stock)
df_feat = df_feat.dropna(subset=['Forward_Log_Return']).reset_index(drop=True)

print(f'Shape after indicator computation: {df_feat.shape}')
print(f'New columns added: {[c for c in df_feat.columns if c not in df.columns]}')

---
## 4. Visualise Indicators — 300 DPI

In [ ]:
# ── Figure 1: RSI — 6 representative stocks ───────────────────────────────────
showcase = ['HBL.KA','OGDC.KA','LUCK.KA','FFC.KA','SYS.KA','MEBL.KA']
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]
    ax.plot(grp['Date'], grp['RSI_14'], color='#9b59b6', linewidth=0.8)
    ax.axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
    ax.axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
    ax.fill_between(grp['Date'], grp['RSI_14'], 70,
                    where=(grp['RSI_14']>=70), alpha=0.15, color='red')
    ax.fill_between(grp['Date'], grp['RSI_14'], 30,
                    where=(grp['RSI_14']<=30), alpha=0.15, color='green')
    ax.set_ylim(0, 100)
    ax.set_title(f'RSI(14) — {ticker.replace(".KA","")}', fontweight='bold')
    ax.set_ylabel('RSI')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=7)

plt.suptitle('Relative Strength Index (RSI-14) — Selected KSE-30 Stocks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig01_RSI.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig01_RSI.png')

In [ ]:
# ── Figure 2: MACD ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]

    ax.plot(grp['Date'], grp['MACD'],        color='#3498db', linewidth=0.8, label='MACD')
    ax.plot(grp['Date'], grp['MACD_Signal'], color='#e74c3c', linewidth=0.8, label='Signal')
    ax.bar(grp['Date'],  grp['MACD_Hist'],
           color=np.where(grp['MACD_Hist'] >= 0, '#2ecc71', '#e74c3c'),
           alpha=0.4, width=1, label='Histogram')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'MACD(12,26,9) — {ticker.replace(".KA","")}', fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=7)

plt.suptitle('MACD Indicator — Selected KSE-30 Stocks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig02_MACD.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig02_MACD.png')

In [ ]:
# ── Figure 3: Bollinger Bands — last 2 years for clarity ─────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    grp = grp[grp['Date'] >= '2023-01-01']   # last 2 years for readability
    ax  = axes[i]

    ax.plot(grp['Date'], grp['Close'],    color='#2c3e50', linewidth=1.0, label='Close')
    ax.plot(grp['Date'], grp['BB_Upper'], color='#e74c3c', linewidth=0.7,
            linestyle='--', label='Upper Band')
    ax.plot(grp['Date'], grp['BB_Mid'],   color='#3498db', linewidth=0.7,
            linestyle='--', label='SMA-20')
    ax.plot(grp['Date'], grp['BB_Lower'], color='#2ecc71', linewidth=0.7,
            linestyle='--', label='Lower Band')
    ax.fill_between(grp['Date'], grp['BB_Upper'], grp['BB_Lower'],
                    alpha=0.08, color='#3498db')
    ax.set_title(f'Bollinger Bands — {ticker.replace(".KA","")} (2023–2025)',
                 fontweight='bold')
    ax.set_ylabel('Price (PKR)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)

plt.suptitle('Bollinger Bands (20,2) — Selected KSE-30 Stocks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig03_Bollinger.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig03_Bollinger.png')

In [ ]:
# ── Figure 4: Momentum (3M, 6M, 12M) ─────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]

    ax.plot(grp['Date'], grp['Mom_3M']*100,  color='#3498db', linewidth=0.8, label='3M Momentum')
    ax.plot(grp['Date'], grp['Mom_6M']*100,  color='#e67e22', linewidth=0.8, label='6M Momentum')
    ax.plot(grp['Date'], grp['Mom_12M']*100, color='#9b59b6', linewidth=0.8, label='12M Momentum')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.fill_between(grp['Date'], grp['Mom_3M']*100, 0,
                    where=(grp['Mom_3M']>=0), alpha=0.08, color='#3498db')
    ax.fill_between(grp['Date'], grp['Mom_3M']*100, 0,
                    where=(grp['Mom_3M']<0),  alpha=0.08, color='#e74c3c')
    ax.set_title(f'Price Momentum — {ticker.replace(".KA","")}', fontweight='bold')
    ax.set_ylabel('Return (%)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=7)

plt.suptitle('Price Momentum (3M / 6M / 12M) — Selected KSE-30 Stocks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig04_Momentum.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig04_Momentum.png')

In [ ]:
# ── Figure 5: Volatility ──────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]

    ax.plot(grp['Date'], grp['Vol_10d']*100, color='#e74c3c', linewidth=0.7, label='10d Vol')
    ax.plot(grp['Date'], grp['Vol_21d']*100, color='#e67e22', linewidth=0.7, label='21d Vol')
    ax.plot(grp['Date'], grp['Vol_63d']*100, color='#9b59b6', linewidth=0.9, label='63d Vol')
    ax.axvline(pd.Timestamp('2020-03-01'), color='black',
               linestyle='--', linewidth=0.7, alpha=0.6)
    ax.set_title(f'Historical Volatility — {ticker.replace(".KA","")}', fontweight='bold')
    ax.set_ylabel('Ann. Volatility (%)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=7)

plt.suptitle('Historical Volatility (10d / 21d / 63d Ann.) — Selected KSE-30 Stocks\n(Dashed = COVID-19)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig05_Volatility.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig05_Volatility.png')

In [ ]:
# ── Figure 6: Rate of Change ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 12))
axes = axes.flatten()

for i, ticker in enumerate(showcase):
    grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]

    ax.plot(grp['Date'], grp['RoC_5d'],  color='#3498db', linewidth=0.7, label='RoC 5d')
    ax.plot(grp['Date'], grp['RoC_10d'], color='#e67e22', linewidth=0.7, label='RoC 10d')
    ax.plot(grp['Date'], grp['RoC_21d'], color='#9b59b6', linewidth=0.9, label='RoC 21d')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'Rate of Change — {ticker.replace(".KA","")}', fontweight='bold')
    ax.set_ylabel('RoC (%)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=7)

plt.suptitle('Rate of Change (5d / 10d / 21d) — Selected KSE-30 Stocks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig06_RoC.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig06_RoC.png')

In [ ]:
# ── Figure 7: Full Indicator Dashboard — Single Stock (HBL) ──────────────────
ticker = 'HBL.KA'
grp = df_feat[df_feat['Ticker']==ticker].sort_values('Date')
grp = grp[grp['Date'] >= '2022-01-01']   # last 3 years

fig, axes = plt.subplots(5, 1, figsize=(18, 22), sharex=True)

# Panel 1: Price + Bollinger
axes[0].plot(grp['Date'], grp['Close'],    color='#2c3e50', lw=1.0, label='Close')
axes[0].plot(grp['Date'], grp['BB_Upper'], color='#e74c3c', lw=0.7, ls='--', label='BB Upper')
axes[0].plot(grp['Date'], grp['BB_Mid'],   color='#3498db', lw=0.7, ls='--', label='BB Mid')
axes[0].plot(grp['Date'], grp['BB_Lower'], color='#2ecc71', lw=0.7, ls='--', label='BB Lower')
axes[0].fill_between(grp['Date'], grp['BB_Upper'], grp['BB_Lower'], alpha=0.07, color='#3498db')
axes[0].set_ylabel('Price (PKR)')
axes[0].set_title(f'{ticker.replace(".KA","")} — Full Indicator Dashboard (2022–2025)', fontweight='bold', fontsize=11)
axes[0].legend(fontsize=7, loc='upper left')

# Panel 2: Volume
axes[1].bar(grp['Date'], grp['Volume'], color='#3498db', alpha=0.5, width=1)
axes[1].set_ylabel('Volume')

# Panel 3: MACD
axes[2].plot(grp['Date'], grp['MACD'],        color='#3498db', lw=0.8, label='MACD')
axes[2].plot(grp['Date'], grp['MACD_Signal'], color='#e74c3c', lw=0.8, label='Signal')
axes[2].bar(grp['Date'],  grp['MACD_Hist'],
            color=np.where(grp['MACD_Hist']>=0,'#2ecc71','#e74c3c'), alpha=0.5, width=1)
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=7)

# Panel 4: RSI
axes[3].plot(grp['Date'], grp['RSI_14'], color='#9b59b6', lw=0.8)
axes[3].axhline(70, color='red',   ls='--', lw=0.8)
axes[3].axhline(30, color='green', ls='--', lw=0.8)
axes[3].fill_between(grp['Date'], grp['RSI_14'], 70, where=(grp['RSI_14']>=70), alpha=0.15, color='red')
axes[3].fill_between(grp['Date'], grp['RSI_14'], 30, where=(grp['RSI_14']<=30), alpha=0.15, color='green')
axes[3].set_ylim(0, 100)
axes[3].set_ylabel('RSI(14)')

# Panel 5: Volatility
axes[4].plot(grp['Date'], grp['Vol_21d']*100, color='#e74c3c', lw=0.8, label='21d Vol')
axes[4].plot(grp['Date'], grp['Vol_63d']*100, color='#9b59b6', lw=0.9, label='63d Vol')
axes[4].set_ylabel('Ann. Vol (%)')
axes[4].set_xlabel('Date')
axes[4].legend(fontsize=7)
axes[4].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[4].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('fig07_Dashboard_HBL.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig07_Dashboard_HBL.png')

---
## 5. Prepare Feature Matrix for Selection
Drop NaN rows (caused by long lookback windows like 252-day momentum)

In [ ]:
# ── Define full feature list ──────────────────────────────────────────────────
FEATURE_COLS = [
    # Price-derived
    'Log_Return', 'RSI_14',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Width', 'BB_Pct',
    'Mom_3M', 'Mom_6M', 'Mom_12M',
    'Vol_10d', 'Vol_21d', 'Vol_63d',
    'RoC_5d', 'RoC_10d', 'RoC_21d',
    'ATR_14',
    'Price_SMA20_Ratio', 'Price_SMA50_Ratio',
    # Volume-derived
    'OBV',
]

TARGET_COL = 'Forward_Log_Return'

# Drop rows with NaN in any feature (warm-up period for long indicators)
df_model = df_feat[['Date','Ticker','Sector'] + FEATURE_COLS + [TARGET_COL]].copy()
before = len(df_model)
df_model = df_model.dropna().reset_index(drop=True)
after  = len(df_model)

print(f'Rows before dropping NaN : {before:,}')
print(f'Rows after  dropping NaN : {after:,}  (dropped {before-after:,} warm-up rows)')
print(f'Features                 : {len(FEATURE_COLS)}')
print(f'Target                   : {TARGET_COL}')
print()
print('Per-stock row counts after warmup:')
print(df_model.groupby('Ticker').size().sort_values())

---
## 6. Feature Selection — CFS (Correlation-Based)
Remove features that are highly correlated with each other (redundant) while keeping features correlated with the target.

In [ ]:
# ── CFS: Step 1 — Feature-Target Correlation ─────────────────────────────────
X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

feat_target_corr = X.corrwith(y).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#2ecc71' if v >= 0.05 else '#e74c3c' for v in feat_target_corr]
feat_target_corr.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0.05, color='black', linestyle='--', linewidth=1, label='Threshold (0.05)')
ax.set_title('Feature–Target Correlation (|r| with Forward Log Return)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('|Correlation|')
ax.set_xlabel('Feature')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('fig08_Feature_Target_Corr.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig08_Feature_Target_Corr.png')
print()
print(feat_target_corr)

In [ ]:
# ── CFS: Step 2 — Inter-Feature Correlation Heatmap ──────────────────────────
feat_corr_matrix = X.corr().abs()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(feat_corr_matrix, dtype=bool))
sns.heatmap(feat_corr_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, mask=mask, linewidths=0.3,
            ax=ax, annot_kws={'size': 7})
ax.set_title('Inter-Feature Correlation Matrix (Absolute Values)',
             fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('fig09_InterFeature_Corr.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig09_InterFeature_Corr.png')

In [ ]:
# ── CFS: Step 3 — Remove Redundant Features ──────────────────────────────────
# Remove features with inter-correlation > 0.85 (keep the one more correlated with target)
INTER_CORR_THRESHOLD = 0.85

corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_cfs = set()
for col in upper.columns:
    correlated_features = upper.index[upper[col] > INTER_CORR_THRESHOLD].tolist()
    for corr_feat in correlated_features:
        # Drop the one less correlated with target
        if feat_target_corr.get(col, 0) >= feat_target_corr.get(corr_feat, 0):
            to_drop_cfs.add(corr_feat)
        else:
            to_drop_cfs.add(col)

features_after_cfs = [f for f in FEATURE_COLS if f not in to_drop_cfs]

print(f'Features before CFS : {len(FEATURE_COLS)}')
print(f'Features dropped    : {len(to_drop_cfs)} → {sorted(to_drop_cfs)}')
print(f'Features after CFS  : {len(features_after_cfs)} → {features_after_cfs}')

---
## 7. Feature Selection — RFE with Random Forest

In [ ]:
# ── RFE with Random Forest ────────────────────────────────────────────────────
# Use a sample for speed (RFE on full 55k rows is slow)
sample_size = min(15000, len(df_model))
df_sample = df_model.sample(sample_size, random_state=42)

X_rfe = df_sample[features_after_cfs].values
y_rfe = df_sample[TARGET_COL].values

# Impute any remaining NaN
imputer = SimpleImputer(strategy='median')
X_rfe   = imputer.fit_transform(X_rfe)

# RFE with Random Forest
N_FEATURES_TO_SELECT = 12

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rfe = RFE(estimator=rf, n_features_to_select=N_FEATURES_TO_SELECT, step=1)
rfe.fit(X_rfe, y_rfe)

# Selected features
rfe_mask     = rfe.support_
rfe_ranking  = rfe.ranking_
FINAL_FEATURES = [f for f, selected in zip(features_after_cfs, rfe_mask) if selected]

print(f'Features selected by RFE : {len(FINAL_FEATURES)}')
print(f'Final feature set        : {FINAL_FEATURES}')

In [ ]:
# ── Feature Importance from Random Forest ────────────────────────────────────
rf_full = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_full.fit(X_rfe, y_rfe)

importance_df = pd.DataFrame({
    'Feature'   : features_after_cfs,
    'Importance': rf_full.feature_importances_,
    'RFE_Selected': rfe_mask
}).sort_values('Importance', ascending=False)

print('=== Feature Importance Ranking ===')
print(importance_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
colors  = ['#2ecc71' if sel else '#bdc3c7' for sel in importance_df['RFE_Selected']]
bars    = ax.barh(importance_df['Feature'], importance_df['Importance'],
                  color=colors, edgecolor='white')
ax.set_title('Random Forest Feature Importance\n(Green = Selected by RFE, Grey = Eliminated)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=7)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Selected by RFE'),
                   Patch(facecolor='#bdc3c7', label='Eliminated by RFE')]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig('fig10_RF_Feature_Importance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig10_RF_Feature_Importance.png')

In [ ]:
# ── RFE Ranking Plot ──────────────────────────────────────────────────────────
ranking_df = pd.DataFrame({
    'Feature': features_after_cfs,
    'RFE_Rank': rfe_ranking
}).sort_values('RFE_Rank')

fig, ax = plt.subplots(figsize=(14, 6))
colors  = ['#2ecc71' if r == 1 else '#e74c3c' for r in ranking_df['RFE_Rank']]
ax.bar(ranking_df['Feature'], ranking_df['RFE_Rank'], color=colors, edgecolor='white')
ax.axhline(1, color='black', linestyle='--', linewidth=1, label='Rank 1 = Selected')
ax.set_title('RFE Feature Rankings\n(Rank 1 = Selected | Higher rank = eliminated earlier)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('RFE Rank')
ax.tick_params(axis='x', rotation=45)
ax.legend()
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Selected (Rank=1)'),
                   Patch(facecolor='#e74c3c', label='Eliminated')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('fig11_RFE_Rankings.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig11_RFE_Rankings.png')

---
## 8. Final Feature Set & Save

In [ ]:
# ── Final feature set summary ─────────────────────────────────────────────────
print('='*55)
print('  FEATURE SELECTION SUMMARY')
print('='*55)
print(f'  Original features        : {len(FEATURE_COLS)}')
print(f'  After CFS (corr > 0.85)  : {len(features_after_cfs)}')
print(f'  After RFE (RF-based)     : {len(FINAL_FEATURES)}')
print()
print('  Final selected features:')
for i, feat in enumerate(FINAL_FEATURES, 1):
    corr_val = feat_target_corr.get(feat, 0)
    imp_val  = importance_df[importance_df['Feature']==feat]['Importance'].values
    imp_str  = f'{imp_val[0]:.4f}' if len(imp_val) > 0 else 'N/A'
    print(f'  {i:2}. {feat:<25} target_corr={corr_val:.4f}  RF_importance={imp_str}')
print('='*55)

In [ ]:
# ── Build final model-ready dataset ──────────────────────────────────────────
KEEP_COLS = ['Date', 'Ticker', 'Sector', 'Close'] + FINAL_FEATURES + [TARGET_COL]
df_final  = df_model[KEEP_COLS].copy().dropna().reset_index(drop=True)

# Normalize final features to [0,1]
scaler_final = MinMaxScaler(feature_range=(0, 1))
df_final[FINAL_FEATURES] = scaler_final.fit_transform(df_final[FINAL_FEATURES])

# Save scaler
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_final, f)

# Save final feature list
with open('final_features.pkl', 'wb') as f:
    pickle.dump(FINAL_FEATURES, f)

print(f'Final model-ready dataset shape: {df_final.shape}')
print(f'Date range: {df_final["Date"].min().date()} → {df_final["Date"].max().date()}')
print(f'Stocks: {df_final["Ticker"].nunique()}')
print(f'Missing values: {df_final.isnull().sum().sum()}')
print()

# 60-day window count
WINDOW = 60
total_windows = sum(len(g) - WINDOW for _, g in df_final.groupby('Ticker') if len(g) > WINDOW)
print(f'Total 60-day windows available: {total_windows:,}')

In [ ]:
# ── Save all output files ─────────────────────────────────────────────────────
df_feat.to_csv('KSE30_all_indicators.csv', index=False)     # all 21 indicators
df_final.to_csv('KSE30_model_ready.csv',   index=False)     # final 12 features, scaled
importance_df.to_csv('feature_importance.csv', index=False) # importance scores

print('Files saved:')
print('  KSE30_all_indicators.csv  — all 21 indicators for all 29 stocks')
print('  KSE30_model_ready.csv     — final selected features, scaled [0,1]')
print('  feature_importance.csv    — RF importance + RFE rankings')
print('  feature_scaler.pkl        — scaler for inverse transform')
print('  final_features.pkl        — list of selected feature names')

In [ ]:
# ── Download all figures and files ────────────────────────────────────────────
from google.colab import files
import glob

print('Downloading all figures (300 DPI)...')
for fig_file in sorted(glob.glob('fig*.png')):
    files.download(fig_file)
    print(f'  Downloaded: {fig_file}')

print()
print('Downloading data files...')
for data_file in ['KSE30_all_indicators.csv','KSE30_model_ready.csv',
                  'feature_importance.csv','feature_scaler.pkl','final_features.pkl']:
    files.download(data_file)
    print(f'  Downloaded: {data_file}')

print()
print('All downloads complete!')

---
## ✅ Feature Engineering Complete

| File | Purpose |
|---|---|
| `KSE30_all_indicators.csv` | All 21 indicators — reference & analysis |
| `KSE30_model_ready.csv` | 12 selected features, scaled — **feed into XGBoost + LSTM** |
| `feature_importance.csv` | RF importance scores + RFE rankings |
| `feature_scaler.pkl` | Scaler for inverse transform |
| `final_features.pkl` | Selected feature names list |

**Figures produced (all 300 DPI):**
- fig01: RSI across 6 stocks
- fig02: MACD across 6 stocks
- fig03: Bollinger Bands across 6 stocks
- fig04: Momentum (3M/6M/12M)
- fig05: Historical Volatility
- fig06: Rate of Change
- fig07: Full Dashboard (HBL)
- fig08: Feature–Target Correlation
- fig09: Inter-Feature Correlation Matrix
- fig10: RF Feature Importance
- fig11: RFE Rankings

**Next → Model Building: XGBoost + LSTM Hybrid**